# Person bounding boxes for MaskSGA

This notebook detects people in images with [DINO-DETR](https://github.com/IDEA-Research/DINO) (DINO-4scale, ResNet-50 backbone, trained on COCO) and saves the person bounding boxes as one JSON file per image folder.

These JSON files are the `--boxes_female` and `--boxes_male` inputs of `generate_masked_images.py`, the first step of the MaskSGA pipeline (see `README.md`).

**Where to run it.** The notebook runs on **Google Colab** or on a **local Linux machine with an NVIDIA GPU**. DINO relies on a custom CUDA operator that is compiled in Section 2, so a GPU is required in both cases. macOS and CPU-only machines are not supported.

**Before you start.** Download the checkpoint `checkpoint0033_4scale.pth` (DINO-4scale, R50, 12 epochs) from the [DINO model zoo](https://drive.google.com/drive/folders/1qD5m1NmK0kjE5hh-G17XUX751WsEG-h_?usp=sharing). On Colab, place it in your Google Drive.

**Contents**

1. Set up the environment *(on Colab, restarts the runtime once)*
2. Build DINO
3. Configuration
4. Load the detector
5. Detect people and save the JSON files
6. Inspect detections *(optional)*

## 1. Set up the environment

DINO's CUDA operator does not build against recent PyTorch and NumPy versions. The notebook requires the versions it was tested with:

| Package | Version |
|---|---|
| `torch` | 2.2.0 (CUDA 12.1 build) |
| `torchvision` | 0.17.0 |
| `numpy` | 1.26.4 |
| `addict`, `yapf`, `timm`, `pycocotools`, `termcolor`, `scipy` | latest (DINO dependencies) |
| `tqdm`, `matplotlib`, `setuptools` | latest |

### On Google Colab

Select a GPU runtime (*Runtime → Change runtime type*; T4, L4 or A100) and run the cell below. It installs the versions above and then **restarts the runtime automatically**, because Colab has already imported its own PyTorch and NumPy versions. Colab may report that the session crashed or restarted; this is expected. Then run the notebook again from the top (*Runtime → Run all*). On the second pass, the cell finds the correct versions and skips the installation.

pip may report dependency conflicts with unrelated preinstalled packages (e.g. `jax`, `opencv`, `torchaudio`). They are not used here and can be ignored.

### On a local machine

To avoid changing your existing environment, the cell does **not** install anything locally. It only checks the versions. Set up a separate environment once, then start Jupyter from it:

```bash
python3.10 -m venv .venv-dino            # Python 3.10 to 3.12
source .venv-dino/bin/activate
pip install torch==2.2.0 torchvision==0.17.0 --index-url https://download.pytorch.org/whl/cu121
pip install numpy==1.26.4 addict yapf timm pycocotools termcolor scipy tqdm matplotlib setuptools notebook
jupyter notebook generate_bounding_boxes.ipynb
```

(With conda, create the environment with `conda create -n masksga-dino python=3.10` and run the same `pip install` commands inside it.)

Compiling the CUDA operator in Section 2 additionally requires:

- **CUDA toolkit 12.x** with `nvcc`, plus a `gcc` version supported by that toolkit. If the toolkit is not installed under `/usr/local/cuda`, set the `CUDA_HOME` environment variable before starting Jupyter.
- **A GPU supported by PyTorch 2.2**, i.e. up to the Hopper generation (e.g. RTX 20/30/40 series, V100, A100, H100). Blackwell GPUs (e.g. RTX 50 series) are not supported by PyTorch 2.2.

In [ ]:
import importlib.metadata as metadata
import os
import shutil
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PINNED = {"torch": "2.2.0", "torchvision": "0.17.0", "numpy": "1.26.4"}
DEPENDENCIES = ["addict", "yapf", "timm", "pycocotools", "termcolor", "scipy", "tqdm", "matplotlib", "setuptools"]
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu121"


def installed_version(package):
    try:
        return metadata.version(package).split("+")[0]
    except metadata.PackageNotFoundError:
        return None


def pip_install(*args):
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:], sep="\n")
        raise RuntimeError("pip install failed (see output above).")


if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU detected. "
        + ("Select a GPU runtime (Runtime -> Change runtime type) and run this cell again."
           if IN_COLAB else "DINO requires an NVIDIA GPU with a working driver.")
    )

wrong_versions = {p: installed_version(p) for p, v in PINNED.items() if installed_version(p) != v}
missing = [p for p in DEPENDENCIES if installed_version(p) is None]

if not (wrong_versions or missing):
    print("Environment ready:",
          ", ".join(f"{p} {installed_version(p)}" for p in [*PINNED, *DEPENDENCIES]))
elif IN_COLAB:
    print("Installing pinned packages. Currently installed:", wrong_versions or "-",
          "| missing:", missing or "-")
    pip_install(f"torch=={PINNED['torch']}", f"torchvision=={PINNED['torchvision']}",
                "--index-url", TORCH_INDEX_URL)
    pip_install(f"numpy=={PINNED['numpy']}", *DEPENDENCIES)
    print("Installation finished. Restarting the runtime now.\n"
          "Afterwards, run the notebook again from the top (Runtime -> Run all).")
    os.kill(os.getpid(), 9)  # restart the Colab runtime
else:
    raise RuntimeError(
        "This environment does not have the required packages.\n"
        f"  Wrong versions: {wrong_versions or '-'} (required: {PINNED})\n"
        f"  Missing: {missing or '-'}\n"
        "Set up a separate environment as described above and start Jupyter from it."
    )

## 2. Build DINO

Clones the DINO repository at a fixed commit and compiles its multi-scale deformable attention operator (`MultiScaleDeformableAttention`) for the current GPU. Compilation takes a few minutes and is skipped if the operator is already installed in this environment.

The repository is cloned to `/content/DINO` on Colab and to `./DINO` (next to this notebook) on a local machine.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

import torch

DINO_DIR = Path("/content/DINO") if IN_COLAB else Path("DINO").resolve()
DINO_REPO = "https://github.com/IDEA-Research/DINO.git"
DINO_COMMIT = "d84a491d41898b3befd8294d1cf2614661fc0953"  # latest commit (July 2024)

if not torch.__version__.startswith("2.2.0"):
    raise RuntimeError(f"torch {torch.__version__} is loaded, expected 2.2.0. "
                       "Complete Section 1 first (on Colab, let the runtime restart).")
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot access a CUDA GPU. Check the GPU and its driver.")

major, minor = torch.cuda.get_device_capability()
if (major, minor) > (9, 0):
    raise RuntimeError(f"{torch.cuda.get_device_name()} (compute capability {major}.{minor}) "
                       "is not supported by PyTorch 2.2. Use a GPU up to the Hopper generation.")


def run(cmd, **kwargs):
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:], sep="\n")
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")


if not DINO_DIR.is_dir():
    run(["git", "clone", "-q", DINO_REPO, str(DINO_DIR)])
    run(["git", "checkout", "-q", DINO_COMMIT], cwd=DINO_DIR)

try:
    import MultiScaleDeformableAttention  # noqa: F401
    print("DINO CUDA operator already built.")
except ImportError:
    from torch.utils.cpp_extension import CUDA_HOME
    if CUDA_HOME is None:
        raise RuntimeError("CUDA toolkit (nvcc) not found, which is needed to compile the "
                           "DINO operator. Install CUDA toolkit 12.x and set CUDA_HOME.")
    print(f"Compiling the DINO CUDA operator for {torch.cuda.get_device_name()} "
          f"(compute capability {major}.{minor}). This takes a few minutes ...")
    run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "."],
        cwd=DINO_DIR / "models" / "dino" / "ops",
        env={**os.environ, "TORCH_CUDA_ARCH_LIST": f"{major}.{minor}"})
    importlib.invalidate_caches()
    import MultiScaleDeformableAttention  # noqa: F401
    print("Done.")

## 3. Configuration

Set the paths below. Each folder in `IMAGE_FOLDERS` produces one file `<OUTPUT_DIR>/<folder name>_boxes.json`. For MaskSGA, run it on one folder of female and one folder of male images per case.

On Colab, your Google Drive is mounted and `DATA_ROOT` points to `/content/drive/MyDrive`. Locally, `DATA_ROOT` is the folder of this notebook. You can also replace the paths with absolute paths.

In the paper, the notebook was run on eight folders: {expressionism, art nouveau, cubism, academicism} × {female, male}.

The detection settings are the ones used in the paper; keep them unchanged to reproduce our results.

In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive")
else:
    DATA_ROOT = Path.cwd()

# DINO-4scale checkpoint from the DINO model zoo
CHECKPOINT_PATH = DATA_ROOT / "checkpoints" / "checkpoint0033_4scale.pth"

# Image folders to process (one JSON file is written per folder)
IMAGE_FOLDERS = [
    DATA_ROOT / "data" / "female_images",
    DATA_ROOT / "data" / "male_images",
]

# Directory for the JSON files
OUTPUT_DIR = DATA_ROOT / "data" / "bounding_boxes"

# Detection settings used in the paper
PERSON_CLASS_ID = 1      # COCO category id of "person"
SUFFICIENT_SCORE = 0.1   # keep person boxes scoring above this value ...
SCORE_MARGIN = 0.5       # ... or above (1 - SCORE_MARGIN) x the image's top person score

# Check the paths before loading the model
if not Path(CHECKPOINT_PATH).is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")
for folder in IMAGE_FOLDERS:
    if not Path(folder).is_dir():
        raise FileNotFoundError(f"Image folder not found: {folder}")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Configuration OK.")

## 4. Load the detector

Builds DINO-4scale from the repository's config file and loads the COCO checkpoint. Warnings about deprecated `timm` imports or the `pretrained` argument come from the DINO code and can be ignored.

In [ ]:
import sys

import torch

# DINO's modules are imported from the repository directory. Note that DINO has a
# local package called `datasets`, which takes precedence over Hugging Face `datasets`.
if str(DINO_DIR) not in sys.path:
    sys.path.insert(0, str(DINO_DIR))

import datasets.transforms as DT  # noqa: E402  (DINO's transforms)
from main import build_model_main  # noqa: E402
from util.slconfig import SLConfig  # noqa: E402

device = torch.device("cuda")

model_config = SLConfig.fromfile(str(DINO_DIR / "config" / "DINO" / "DINO_4scale.py"))
model_config.device = "cuda"
model, _, postprocessors = build_model_main(model_config)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
model.load_state_dict(checkpoint["model"])
model = model.to(device).eval()
print("DINO-4scale loaded.")

## 5. Detect people and save the JSON files

Each image is resized so that its shorter side is 800 px (longer side at most 1333 px) and normalised with the ImageNet statistics, as in DINO's COCO evaluation. Among DINO's top 300 predictions, a box is kept if it is labelled *person* and its score is above `SUFFICIENT_SCORE` **or** above `(1 - SCORE_MARGIN)` times the highest person score in that image.

Each JSON file maps image file names to their detections. Box coordinates are `[x1, y1, x2, y2]`, relative to the image width and height (range 0 to 1):

```json
{
    "image_name.jpg": {
        "scores": [0.83, 0.41],
        "boxes_xyxy": [[0.12, 0.05, 0.64, 0.98], [0.55, 0.20, 0.91, 0.97]],
        "n_boxes": 2
    }
}
```

Images in which no person is detected get empty lists and `"n_boxes": 0`.

**Relation to `generate_masked_images.py`.** The JSON files store a permissive set of boxes. `generate_masked_images.py` filters them again with stricter thresholds when drawing the masks (`--score_threshold 0.3 --score_margin 0.3` by default), so the boxes stored here are a superset of the boxes used for masking.

In [ ]:
import json
from pathlib import Path

from PIL import Image
from tqdm.auto import tqdm

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# With a target size of 1 x 1, the post-processor returns box coordinates in [0, 1].
RELATIVE_SIZE = torch.tensor([[1.0, 1.0]], device=device)


def preprocess(pil_image):
    # Resize and normalise as in DINO's COCO evaluation (a single target size, so deterministic).
    target = {"boxes": torch.zeros(0, 4), "labels": torch.zeros(0)}
    image, target = DT.RandomResize([800], max_size=1333)(pil_image, target)
    image, target = DT.ToTensor()(image, target)
    image, _ = DT.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])(image, target)
    return image


@torch.no_grad()
def detect_people(pil_image):
    # Run DINO on one RGB image and return the selected person boxes.
    image = preprocess(pil_image).to(device)
    output = postprocessors["bbox"](model(image[None]), RELATIVE_SIZE)[0]

    person_mask = output["labels"] == PERSON_CLASS_ID
    person_scores = output["scores"][person_mask]
    if person_scores.numel() == 0:
        return {"scores": [], "boxes_xyxy": [], "n_boxes": 0}

    min_score = min(SUFFICIENT_SCORE, (1 - SCORE_MARGIN) * person_scores.max())
    keep = person_mask & (output["scores"] > min_score)
    return {
        "scores": output["scores"][keep].tolist(),
        "boxes_xyxy": output["boxes"][keep].tolist(),
        "n_boxes": int(keep.sum().item()),
    }


def process_folder(folder, output_dir):
    # Detect people in every image of `folder` and write <folder name>_boxes.json.
    folder = Path(folder)
    image_paths = sorted(p for p in folder.iterdir()
                         if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

    results, no_person, failed = {}, [], []
    for image_path in tqdm(image_paths, desc=folder.name):
        try:
            with Image.open(image_path) as opened:
                detections = detect_people(opened.convert("RGB"))
        except Exception as exc:
            failed.append(f"{image_path.name} ({exc})")
            continue
        results[image_path.name] = detections
        if detections["n_boxes"] == 0:
            no_person.append(image_path.name)

    output_path = Path(output_dir) / f"{folder.name}_boxes.json"
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(results, f, indent=4)

    print(f"{folder.name}: {len(results)} images -> {output_path}")
    if no_person:
        print(f"  no person detected in {len(no_person)} image(s): {', '.join(no_person)}")
    if failed:
        print(f"  failed to process {len(failed)} image(s): {', '.join(failed)}")
    return output_path

In [ ]:
for folder in IMAGE_FOLDERS:
    process_folder(folder, OUTPUT_DIR)

## 6. Inspect detections (optional)

Draws the stored boxes on a few random images from one folder. Boxes that `generate_masked_images.py` will mask with its default thresholds are drawn in red; additional stored boxes that it will ignore are drawn as dashed yellow lines.

In [ ]:
import random

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

PREVIEW_INDEX = 0            # which entry of IMAGE_FOLDERS to preview
N_PREVIEW = 6
MASK_SCORE_THRESHOLD = 0.3   # defaults of generate_masked_images.py
MASK_SCORE_MARGIN = 0.3

folder = Path(IMAGE_FOLDERS[PREVIEW_INDEX])
with open(Path(OUTPUT_DIR) / f"{folder.name}_boxes.json", encoding="utf-8") as f:
    detections = json.load(f)

names = random.Random(0).sample(sorted(detections), min(N_PREVIEW, len(detections)))
fig, axes = plt.subplots(1, len(names), figsize=(4 * len(names), 4), squeeze=False)

for ax, name in zip(axes[0], names):
    with Image.open(folder / name) as opened:
        image = opened.convert("RGB")
    width, height = image.size
    ax.imshow(image)

    info = detections[name]
    if info["scores"]:
        threshold = min(MASK_SCORE_THRESHOLD, (1 - MASK_SCORE_MARGIN) * max(info["scores"]))
        for (x1, y1, x2, y2), score in zip(info["boxes_xyxy"], info["scores"]):
            masked = score >= threshold
            ax.add_patch(Rectangle(
                (x1 * width, y1 * height), (x2 - x1) * width, (y2 - y1) * height,
                fill=False, linewidth=2,
                edgecolor="red" if masked else "yellow",
                linestyle="-" if masked else "--",
            ))
    ax.set_title(f"{name[:30]}\n{info['n_boxes']} stored box(es)", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()